## Fase 1.1: Montaje de Drive e Ingesta de Datos
En esta sección conectamos el entorno de Google Colab con Google Drive para acceder al dataset maestro. Se realiza una lectura multianual de las pestañas '2023', '2024' y '2025' para consolidar la base transaccional.

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import os
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Configuración de rutas y variables
PATH_PROYECTO = '/content/drive/MyDrive/Proyecto_MIA'
FILE_NAME = 'Archivo de ventas 2023-2024-2025.xlsx'
FILE_PATH = os.path.join(PATH_PROYECTO, FILE_NAME)

PESTAÑAS = ['2023', '2024', '2025']
COL_SENSIBLES = ['RUC_CLIENTE', 'NOMBRE_CLIENTE', 'MAIL', 'TELEFONO']
COL_FECHA = 'FECHADOCUMENTO'

def anonimizar_sha256(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "N/A"
    return hashlib.sha256(str(texto).encode()).hexdigest()

print("Iniciando Pipeline de Ingesta...")
hojas = []
for sheet in PESTAÑAS:
    df_sheet = pd.read_excel(FILE_PATH, sheet_name=sheet)
    hojas.append(df_sheet)

df_raw = pd.concat(hojas, ignore_index=True)

# Anonimización Ética (LOPDP)
print("Anonimizando datos sensibles...")
for col in COL_SENSIBLES:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].apply(anonimizar_sha256)

# Limpieza Básica
df_raw[COL_FECHA] = pd.to_datetime(df_raw[COL_FECHA], errors='coerce')
df_raw = df_raw.dropna(subset=[COL_FECHA]) # Eliminar registros sin fecha

print("-" * 30)
print(f"Dataset Maestro Creado: {df_raw.shape[0]} registros.")
print(f"Ventas Negativas: {(df_raw['VENTA'] < 0).sum()}")
print("-" * 30)

Mounted at /content/drive
Iniciando Pipeline de Ingesta...
Anonimizando datos sensibles...
------------------------------
Dataset Maestro Creado: 35822 registros.
Ventas Negativas: 990
------------------------------


In [ ]:
from google.colab import files

# Definir nombre del nuevo dataset
OUTPUT_NAME = 'Swarovski_Dataset_Limpio_Anonimizado.csv'
OUTPUT_PATH = os.path.join(PATH_PROYECTO, OUTPUT_NAME)

# Guardar en Drive (formato CSV para mayor compatibilidad y velocidad)
df_raw.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"Archivo guardado en Drive: {OUTPUT_PATH}")

# Descarga directa al ordenador
files.download(OUTPUT_PATH)

Archivo guardado en Drive: /content/drive/MyDrive/Proyecto_MIA/Swarovski_Dataset_Limpio_Anonimizado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>